# 03 · Graph density, saturation & operating point

Addresses **Reviewer 1.1/1.3/1.6/1.9** and **Reviewer 3.5 (descriptive)** using the
master prediction table from notebook 01 (held-out validation patients).
- graph-density (hop / neighbourhood-size) sweep
- images-per-patient saturation
- F1-optimal confidence threshold
- per-centre breakdown

In [ ]:
# --- notebook preamble: locate the reusable analysis library ------------------
import os, sys, warnings
warnings.filterwarnings("ignore")
LIB = os.path.join(os.path.dirname(os.path.abspath("__file__")), "lib")
if not os.path.isdir(LIB):
    LIB = os.path.join(os.getcwd(), "lib")
sys.path.insert(0, LIB)

# _lib_raman defines ROOT / DATA / MODEL_PATH / META_PATH (absolute paths to the
# original project). If you moved the data, override them here, e.g.:
#   import _lib_raman as L; L.ROOT = "/path/to/Project_Raman"; L.DATA = ...
import _lib_raman as L
print("Project ROOT :", L.ROOT)
print("Graph DATA   :", L.DATA)
print("Model        :", L.MODEL_PATH)
print("Classes      :", L.CLASSES)

import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, f1_score
df = pd.read_csv('outputs/master_predictions_meta.csv')
val = df[df.split=='val'].copy()
print('held-out val rows:', len(val), '| patients:', val.patient_id.nunique())

## 1 · Graph-density (hop) sweep

In [ ]:
def per_class_auc(d):
    yt=d['y_true'].values; out={}
    for ci,c in enumerate(L.CLASSES):
        yb=(yt==ci).astype(int)
        out[c]=roc_auc_score(yb, d[f'logit_{c}'].values) if len(np.unique(yb))>1 else np.nan
    return out
rows=[]
for (gs,hop),g in val.groupby(['graph_set','hop']):
    a=per_class_auc(g)
    rows.append(dict(graph_set=gs,hop=hop,median_nodes=g.n_nodes.median(),
                     n=len(g),mean_auc=np.nanmean(list(a.values())),
                     bal_acc=balanced_accuracy_score(g.y_true,g.y_pred)))
dens=pd.DataFrame(rows).sort_values('median_nodes'); dens.round(3)

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(4.5,3))
ax.plot(dens.median_nodes, dens.mean_auc,'-o',color='#0072B2')
ax.set_xlabel('Median subgraph size (nodes)'); ax.set_ylabel('Mean AUC')
ax.set_title('Performance vs graph density'); plt.show()
print('peak:', dens.loc[dens.mean_auc.idxmax(),['graph_set','hop','median_nodes','mean_auc']].to_dict())

## 2 · Images-per-patient saturation (hop-3)

In [ ]:
v3 = val[(val.graph_set=='small') & (val.hop==3)].copy()
score_cols=[f'logit_{c}' for c in L.CLASSES]
img = v3.groupby(['patient_id','image_id']).agg({**{c:'mean' for c in score_cols},'y_true':'first'}).reset_index()
rng=np.random.RandomState(42); sat=[]
for k in range(1,9):
    accs=[]
    for _ in range(30):
        preds=[];trues=[]
        for pid,grp in img.groupby('patient_id'):
            take=grp.sample(min(k,len(grp)), random_state=rng.randint(1_000_000))
            preds.append(int(np.argmax(take[score_cols].mean().values))); trues.append(int(take['y_true'].iloc[0]))
        accs.append(accuracy_score(trues,preds))
    sat.append(dict(n_images=k, acc=np.mean(accs), std=np.std(accs)))
sat=pd.DataFrame(sat); sat.round(3)

In [ ]:
fig,ax=plt.subplots(figsize=(4.5,3))
ax.errorbar(sat.n_images, sat.acc, yerr=sat['std'], fmt='-o', capsize=3, color='#0072B2')
ax.axvline(4, ls='--', color='grey'); ax.set_xlabel('Images per patient'); ax.set_ylabel('Patient accuracy')
ax.set_title('Sampling saturation'); plt.show()

## 3 · Confidence-threshold trade-off & per-centre

In [ ]:
thr=[]
for t in np.round(np.arange(0.30,0.55,0.05),2):
    hc=v3[v3.conf>=t]
    if len(hc)<5: continue
    thr.append(dict(threshold=t, coverage=len(hc)/len(v3), accuracy=accuracy_score(hc.y_true,hc.y_pred)))
print(pd.DataFrame(thr).round(3).to_string(index=False))
cen=[]
for c,g in v3.groupby('center'):
    a=per_class_auc(g)
    cen.append(dict(center=c, n_pat=g.patient_id.nunique(), mean_auc=np.nanmean(list(a.values())),
                    accuracy=accuracy_score(g.y_true,g.y_pred)))
pd.DataFrame(cen).round(3)